# Recirculating Water System - Pump Head and Power Analysis
Computes required pump head and power for flow rates from 10 to 90 GPM.

In [8]:
from dataclasses import dataclass
import math

G = 32.174          # ft/s^2
GAMMA = 62.4        # lb/ft^3
NU_WATER = 1.076e-5 # ft^2/s, water near room temperature


@dataclass
class SystemParams:
    z1_ft: float = 0.0
    z2_ft: float = 50.0 / 12.0
    L_main_ft: float = 320.0 / 12.0
    D_main_ft: float = 2.0 / 12.0
    D_small_ft: float = 1.0 / 12.0
    roughness_ft: float = 5.0e-6
    n_elbows: int = 5
    K_elbow: float = 0.90
    K_entrance: float = 0.50
    K_contraction: float = 0.40
    K_expansion: float | None = None
    efficiency: float = 0.65


def flow_rate_conversion(gpm: float) -> float:
    """Convert GPM to ft^3/s."""
    return gpm * 0.133681 / 60.0


def area(d_ft: float) -> float:
    return math.pi * d_ft**2 / 4.0


def velocity(Q_cfs: float, d_ft: float) -> float:
    return Q_cfs / area(d_ft)


def reynolds_number(v: float, d_ft: float, nu: float = NU_WATER) -> float:
    return v * d_ft / nu


def friction_factor(Re: float, eps_ft: float, d_ft: float) -> float:
    """Darcy friction factor. Laminar fallback, Swamee-Jain otherwise."""
    if Re <= 0.0:
        return 0.0
    if Re < 2000.0:
        return 64.0 / Re
    return 0.25 / (math.log10(eps_ft / (3.7 * d_ft) + 5.74 / (Re**0.9)) ** 2)


def velocity_head(v: float, g: float = G) -> float:
    return v**2 / (2.0 * g)


def major_losses(f: float, L_ft: float, d_ft: float, v: float, g: float = G) -> float:
    return f * (L_ft / d_ft) * velocity_head(v, g)


def minor_losses(
    v_main: float,
    v_small: float,
    n_elbows: int,
    K_elbow: float,
    K_entrance: float,
    K_contraction: float,
    K_expansion: float,
    g: float = G,
) -> float:
    h_main = (K_entrance + n_elbows * K_elbow) * velocity_head(v_main, g)
    h_small = (K_contraction + K_expansion) * velocity_head(v_small, g)
    return h_main + h_small


def pump_head(z1: float, z2: float, v1_point: float, v2_point: float, hL: float, g: float = G) -> float:
    return (z2 - z1) + (v2_point**2 - v1_point**2) / (2.0 * g) + hL


def pump_power(Q_cfs: float, hs_ft: float, efficiency: float, gamma: float = GAMMA) -> float:
    return gamma * Q_cfs * hs_ft / (550.0 * efficiency)


def solve_system(flow_rates_gpm, p: SystemParams):
    A_main = area(p.D_main_ft)
    A_small = area(p.D_small_ft)
    K_exp = p.K_expansion if p.K_expansion is not None else (1.0 - A_small / A_main) ** 2

    results = []

    for gpm in flow_rates_gpm:
        Q = flow_rate_conversion(gpm)

        v_main = velocity(Q, p.D_main_ft)   # 2 in pipe
        v_small = velocity(Q, p.D_small_ft) # 1 in section

        Re = reynolds_number(v_main, p.D_main_ft)
        f = friction_factor(Re, p.roughness_ft, p.D_main_ft)

        h_major = major_losses(f, p.L_main_ft, p.D_main_ft, v_main)
        h_minor = minor_losses(
            v_main=v_main,
            v_small=v_small,
            n_elbows=p.n_elbows,
            K_elbow=p.K_elbow,
            K_entrance=p.K_entrance,
            K_contraction=p.K_contraction,
            K_expansion=K_exp
        )

        hL = h_major + h_minor
        hs = pump_head(p.z1_ft, p.z2_ft, 0.0, v_main, hL)
        hp = pump_power(Q, hs, p.efficiency)

        results.append({
            "GPM": gpm,
            "Q_cfs": Q,
            "Velocity_ft_s": v_main,
            "Re": Re,
            "f": f,
            "MajorLoss_ft": h_major,
            "MinorLoss_ft": h_minor,
            "TotalHeadLoss_ft": hL,
            "PumpHead_ft": hs,
            "PumpPower_hp": hp
        })

    return results


def print_table(results):
    header = (
        f"{'GPM':>5} {'V(ft/s)':>10} {'Re':>12} {'f':>8} "
        f"{'Major(ft)':>12} {'Minor(ft)':>12} {'hL(ft)':>10} "
        f"{'hs(ft)':>10} {'HP':>10}"
    )
    print(header)
    print("-" * len(header))

    for r in results:
        print(
            f"{r['GPM']:5.0f} "
            f"{r['Velocity_ft_s']:10.3f} "
            f"{r['Re']:12.0f} "
            f"{r['f']:8.4f} "
            f"{r['MajorLoss_ft']:12.3f} "
            f"{r['MinorLoss_ft']:12.3f} "
            f"{r['TotalHeadLoss_ft']:10.3f} "
            f"{r['PumpHead_ft']:10.3f} "
            f"{r['PumpPower_hp']:10.3f}"
        )


if __name__ == "__main__":
    params = SystemParams()

    # Flow rates: 10 to 90 GPM
    flow_rates = range(10, 100, 10)

    print("=" * 95)
    print("Case 1 — 1.0-inch pump inlet/outlet piping")
    print("=" * 95)
    results = solve_system(flow_rates, params)
    print_table(results)

Case 1 — 1.0-inch pump inlet/outlet piping
  GPM    V(ft/s)           Re        f    Major(ft)    Minor(ft)     hL(ft)     hs(ft)         HP
-------------------------------------------------------------------------------------------------
   10      1.021        15819   0.0275        0.071        0.331      0.402      4.585      0.018
   20      2.042        31637   0.0232        0.240        1.323      1.563      5.794      0.045
   30      3.064        47456   0.0211        0.493        2.976      3.469      7.781      0.091
   40      4.085        63274   0.0198        0.823        5.290      6.114     10.540      0.164
   50      5.106        79093   0.0189        1.227        8.266      9.493     14.065      0.273
   60      6.127        94911   0.0182        1.703       11.903     13.606     18.356      0.428
   70      7.149       110730   0.0177        2.247       16.201     18.448     23.409      0.637
   80      8.170       126549   0.0172        2.859       21.161     24.020

## Case 2: 1.5-inch Pump Inlet/Outlet Piping

Same recirculating system as above, but the pipe section at the pump inlet and outlet is **1.5 inches** (instead of 1 inch).

**Changes from the original case:**
- `D_small` = 1.5 in (pump connection pipe diameter)
- **Contraction** (2 in → 1.5 in): area ratio A₂/A₁ = (1.5/2)² = 0.5625 → K_contraction ≈ 0.25 (reduced from 0.40)
- **Expansion** (1.5 in → 2 in): computed automatically via Borda-Carnot: K = (1 − A_small/A_main)² ≈ 0.191
- All other parameters (pipe length, elbows, roughness, efficiency, elevations) remain identical.

In [5]:
# ── Case 2: 1.5-inch pump connection ─────────────────────────────────────────
# System is identical to Case 1 except the pump inlet/outlet pipe is 1.5 in.
#
# K value rationale (referenced to the smaller-pipe velocity):
#   Contraction 2" → 1.5":  area ratio A2/A1 = (1.5/2)^2 = 0.5625
#                            K ≈ 0.25  (vs. 0.40 for 2"→1")
#   Expansion  1.5" → 2":   Borda-Carnot auto-computed:
#                            K = (1 - A_small/A_main)^2 ≈ 0.191
# All other parameters (lengths, elbows, roughness, efficiency, elevations)
# are unchanged.

params_1p5in = SystemParams(
    z1_ft         = 0.0,
    z2_ft         = 50.0 / 12.0,
    L_main_ft     = 320.0 / 12.0,
    D_main_ft     = 2.0 / 12.0,          # 2-inch main pipe (unchanged)
    D_small_ft    = 1.5 / 12.0,          # 1.5-inch pump connection
    roughness_ft  = 5.0e-6,
    n_elbows      = 5,
    K_elbow       = 0.90,
    K_entrance    = 0.50,
    K_contraction = 0.25,                # 2" → 1.5" (reduced from 0.40)
    K_expansion   = None,                # auto: Borda-Carnot (1 - A_small/A_main)^2
    efficiency    = 0.65,
)

flow_rates = range(10, 100, 10)

print("=" * 95)
print("Case 2 — 1.5-inch pump inlet/outlet piping")
print("=" * 95)
results_1p5in = solve_system(flow_rates, params_1p5in)
print_table(results_1p5in)

Case 2 — 1.5-inch pump inlet/outlet piping
  GPM    V(ft/s)           Re        f    Major(ft)    Minor(ft)     hL(ft)     hs(ft)         HP
-------------------------------------------------------------------------------------------------
   10      1.021        15819   0.0275        0.071        0.104      0.175      4.358      0.017
   20      2.042        31637   0.0232        0.240        0.415      0.655      4.886      0.038
   30      3.064        47456   0.0211        0.493        0.933      1.426      5.738      0.067
   40      4.085        63274   0.0198        0.823        1.658      2.482      6.908      0.107
   50      5.106        79093   0.0189        1.227        2.591      3.819      8.390      0.163
   60      6.127        94911   0.0182        1.703        3.731      5.434     10.184      0.238
   70      7.149       110730   0.0177        2.247        5.079      7.326     12.287      0.334
   80      8.170       126549   0.0172        2.859        6.634      9.492

## Case 3: 1.25-inch Pump Inlet/Outlet Piping

Same recirculating system, but the pump connection pipe is **1.25 inches**.

**Changes from Case 1 (1-inch pump connection):**
- `D_small` = 1.25 in
- **Contraction** (2 in → 1.25 in): area ratio A₂/A₁ = (1.25/2)² = 0.391 → K_contraction ≈ 0.34  
  *(via Idelchik: K = 0.5·(1−AR)^¾; reproduces 0.40 for 1" and 0.25 for 1.5")*
- **Expansion** (1.25 in → 2 in): Borda-Carnot auto-computed: K = (1 − 0.391)² ≈ 0.371
- All other parameters unchanged.

In [7]:
# ── Case 3: 1.25-inch pump connection ────────────────────────────────────────
# K value rationale:
#   Contraction 2" → 1.25":  AR = (1.25/2)^2 = 0.391
#                             K = 0.5*(1 - AR)^0.75 = 0.5*(0.609)^0.75 ≈ 0.34
#                             (Idelchik; same formula gives 0.40 for 1" and 0.25 for 1.5")
#   Expansion  1.25" → 2":   Borda-Carnot auto-computed:
#                             K = (1 - A_small/A_main)^2 = (1 - 0.391)^2 ≈ 0.371

params_1p25in = SystemParams(
    z1_ft         = 0.0,
    z2_ft         = 50.0 / 12.0,
    L_main_ft     = 320.0 / 12.0,
    D_main_ft     = 2.0 / 12.0,          # 2-inch main pipe (unchanged)
    D_small_ft    = 1.25 / 12.0,         # 1.25-inch pump connection
    roughness_ft  = 5.0e-6,
    n_elbows      = 5,
    K_elbow       = 0.90,
    K_entrance    = 0.50,
    K_contraction = 0.34,                # 2" → 1.25" (Idelchik formula)
    K_expansion   = None,                # auto: Borda-Carnot (1 - A_small/A_main)^2
    efficiency    = 0.65,
)

flow_rates = range(10, 100, 10)

print("=" * 95)
print("Case 3 — 1.25-inch pump inlet/outlet piping")
print("=" * 95)
results_1p25in = solve_system(flow_rates, params_1p25in)
print_table(results_1p25in)

Case 3 — 1.25-inch pump inlet/outlet piping
  GPM    V(ft/s)           Re        f    Major(ft)    Minor(ft)     hL(ft)     hs(ft)         HP
-------------------------------------------------------------------------------------------------
   10      1.021        15819   0.0275        0.071        0.157      0.228      4.411      0.017
   20      2.042        31637   0.0232        0.240        0.626      0.867      5.098      0.040
   30      3.064        47456   0.0211        0.493        1.409      1.902      6.215      0.073
   40      4.085        63274   0.0198        0.823        2.506      3.329      7.755      0.121
   50      5.106        79093   0.0189        1.227        3.915      5.142      9.714      0.189
   60      6.127        94911   0.0182        1.703        5.638      7.340     12.090      0.282
   70      7.149       110730   0.0177        2.247        7.673      9.920     14.881      0.405
   80      8.170       126549   0.0172        2.859       10.022     12.88